# Notebook 13 - Experiment 21: Confidence-Aware Ethical Risk
### Novelty 4
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

The standard fairness metrics count how often the model is wrong for each subgroup,
but none of them look at how *confident* it is when it's wrong. That gap is what this
experiment is about.

It matters in practice. Someone acting on a prediction made at 0.90 confidence trusts
it far more than one at 0.55. If the model's high-confidence mistakes pile up in the
sarcasm subgroup — which is also where it's least accurate — then it's most sure of
itself exactly where it's worst. That's the risk my thesis title points at, and plain
accuracy numbers don't show it.

I use two measures per subgroup:
- **HCER** — the share of that subgroup's errors made above 0.70 confidence
- **MCE** — the mean confidence across all of that subgroup's wrong predictions

No training, about fifteen minutes. Fills **Table 10**.

## Cell 1: Setup

In [ ]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.5 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal


## Cell 2: Load Predictions

Everything needed is already in the saved file: the confidence column was written
when Experiment 10 ran.

In [ ]:
pred = load_predictions("exp10", "FinalModel")

print(f"Test predictions : {len(pred):,}")
print(f"Errors           : {(pred['correct'] == 0).sum():,}")
print(f"Overall accuracy : {pred['correct'].mean():.4f}")
print(f"\nConfidence threshold for HCER: {HCER_CONFIDENCE_THRESHOLD}")
print("\nConfidence distribution:")
print(pred["confidence"].describe().round(4).to_string())

Test predictions : 12,284
Errors           : 4,937
Overall accuracy : 0.5981

Confidence threshold for HCER: 0.7

Confidence distribution:
count    12284.0000
mean         0.7282
std          0.1579
min          0.3397
25%          0.5949
50%          0.7329
75%          0.8643
max          1.0000


## Cell 3: Compute HCER and MCE

In [ ]:
conf = confidence_risk(pred)

print("="*80)
print("CONFIDENCE-AWARE RISK BY SUBGROUP")
print("="*80)
print(conf.round(4).to_string(index=False))

flagged = conf[(conf["HCER"] > 0.30) & (conf["Subgroup"] != "other")]
if len(flagged):
    print(f"\n{len(flagged)} subgroup(s) exceed the 0.30 HCER threshold:")
    for _, r in flagged.iterrows():
        print(f"  {r['Subgroup']}: HCER {r['HCER']:.3f} — "
              f"{r['HCER']*100:.0f}% of its errors were made confidently")
else:
    print("\nNo subgroup exceeds the 0.30 HCER threshold.")

CONFIDENCE-AWARE RISK BY SUBGROUP
   Subgroup  N Misclassified  Misclassification Rate   HCER    MCE HCER > 0.30?
emoji-heavy              264                  0.3793 0.4962 0.6981          Yes
      other              405                  0.3629 0.4889 0.6959          Yes
slang-heavy               23                  0.3833 0.4783 0.7064          Yes
     formal             4242                  0.4080 0.4382 0.6809          Yes
    sarcasm                3                  0.2143 0.3333 0.5994          Yes

4 subgroup(s) exceed the 0.30 HCER threshold:
  emoji-heavy: HCER 0.496 — 50% of its errors were made confidently
  slang-heavy: HCER 0.478 — 48% of its errors were made confidently
  formal: HCER 0.438 — 44% of its errors were made confidently
  sarcasm: HCER 0.333 — 33% of its errors were made confidently


## Cell 4: The Overconfidence Test

The specific pattern this experiment is looking for: a subgroup where the error
rate is above average **and** the confidence on those errors is also above
average. Those two things together are what constitutes confident
misinformation.

In [ ]:
real = conf[conf["Subgroup"] != "other"].copy()

if REFERENCE_SUBGROUP in real["Subgroup"].values:
    ref = real[real["Subgroup"] == REFERENCE_SUBGROUP].iloc[0]
    print(f"Reference group ({REFERENCE_SUBGROUP}):")
    print(f"  error rate {ref['Misclassification Rate']:.4f}   "
          f"HCER {ref['HCER']:.4f}   MCE {ref['MCE']:.4f}\n")

    rows = []
    for _, r in real.iterrows():
        if r["Subgroup"] == REFERENCE_SUBGROUP:
            continue
        err_gap  = r["Misclassification Rate"] - ref["Misclassification Rate"]
        hcer_gap = r["HCER"] - ref["HCER"]
        overconf = (err_gap > 0) and (hcer_gap > 0)
        rows.append({
            "Subgroup":                     r["Subgroup"],
            "Error Rate Gap":               err_gap,
            "HCER Gap":                     hcer_gap,
            "MCE Gap":                      r["MCE"] - ref["MCE"],
            "Overconfident Where Weakest?": "YES" if overconf else "No",
        })
        print(f"{r['Subgroup']}:")
        print(f"  error rate gap {err_gap:+.4f}   HCER gap {hcer_gap:+.4f}")
        if overconf:
            print("  -> More errors AND more confident about them.")
            print("     This is confident misinformation about this group.")
        else:
            print("  -> Not both worse and more confident.")
        print()

    overconf_df = pd.DataFrame(rows)
else:
    print("Reference subgroup missing — cannot compute gaps.")
    overconf_df = pd.DataFrame()

Reference group (formal):
  error rate 0.4080   HCER 0.4382   MCE 0.6809

emoji-heavy:
  error rate gap -0.0287   HCER gap +0.0580
  -> Not both worse and more confident.

slang-heavy:
  error rate gap -0.0246   HCER gap +0.0400
  -> Not both worse and more confident.

sarcasm:
  error rate gap -0.1937   HCER gap -0.1049
  -> Not both worse and more confident.



## Cell 5: Worst Individual Cases

The specific posts where the model was most certain and most wrong. These are
worth quoting directly in the discussion chapter.

In [ ]:
D = PATHS["data"]
try:
    mis = pd.read_parquet(D / "misclassified_instances.parquet")
    worst = mis.nlargest(10, "confidence")

    print("TEN MOST CONFIDENT ERRORS")
    print("="*90)
    for _, r in worst.iterrows():
        print(f"[{r['subgroup']:<12}] {r['y_true']} -> {r['y_pred']}  "
              f"confidence {r['confidence']:.3f}")
        print(f"    {str(r['text'])[:100]}")
    print("\nA prediction at 0.95 confidence that is wrong is worse in")
    print("deployment than one at 0.55 that is wrong, because it will be acted on.")
except FileNotFoundError:
    print("misclassified_instances.parquet not found. Run Notebook 8 first.")

TEN MOST CONFIDENT ERRORS
[other       ] neutral -> positive  confidence 0.998
    @user perfect @user
[other       ] neutral -> positive  confidence 0.997
    Happy @user oppression day
[formal      ] neutral -> positive  confidence 0.994
    @user When will Melania do her "I have a dream" speech? I'm looking forward to it :)
[formal      ] positive -> neutral  confidence 0.994
    The Reputation Doctor weighs in on Tony Romo #NFL @user joins @user on #TheMorningRush LISTEN:
[other       ] neutral -> positive  confidence 0.994
    @user @user @user good mans
[formal      ] negative -> positive  confidence 0.992
    Is this the best George soros could come up lots of love
[other       ] neutral -> positive  confidence 0.992
    AND ALIENS!!!!!1 AND ROBOTS!!!!!!!! SPACE ROBOTS!!!!!!!!!!
[other       ] neutral -> negative  confidence 0.992
    Cubs fuck horny ass cougars
[formal      ] neutral -> positive  confidence 0.991
    Bannon(https://t.co/yEnY1JX0BI): "It's the greatest opportuni

## Cell 6: Table 10

In [ ]:
table10 = conf.copy()
if not overconf_df.empty:
    table10 = table10.merge(overconf_df, on="Subgroup", how="left")

def interpret(r):
    if r["Subgroup"] == REFERENCE_SUBGROUP:
        return "Reference group."
    if r["HCER"] > 0.30:
        return (f"{r['HCER']*100:.0f}% of errors made at confidence above 0.70. "
                f"High ethical risk — the model is confidently wrong here.")
    if r["HCER"] > 0.20:
        return f"Moderate confident-error rate at {r['HCER']*100:.0f}%."
    return "Errors are mostly low-confidence, which is the safer failure mode."

table10["Interpretation"] = table10.apply(interpret, axis=1)

print("="*100)
print("TABLE 10 — CONFIDENCE-AWARE ETHICAL RISK")
print("="*100)
print(table10.round(4).to_string(index=False))

max_hcer = table10[table10["Subgroup"] != "other"]["HCER"].max()
score = score_misclassification_risk(max_hcer)
print(f"\nMax HCER across subgroups : {max_hcer:.4f}")
print(f"Misclassification Risk    : {score} / 3")

save_result_table(table10, "Table10_Confidence_Risk")
print("\nNovelty 4 complete. Feeds the Misclassification Risk dimension of Table 6.")

TABLE 10 — CONFIDENCE-AWARE ETHICAL RISK
   Subgroup  N Misclassified  Misclassification Rate   HCER    MCE HCER > 0.30?  Error Rate Gap  HCER Gap  MCE Gap Overconfident Where Weakest?                                                                                        Interpretation
emoji-heavy              264                  0.3793 0.4962 0.6981          Yes         -0.0287    0.0580   0.0172                           No 50% of errors made at confidence above 0.70. High ethical risk — the model is confidently wrong here.
      other              405                  0.3629 0.4889 0.6959          Yes             NaN       NaN      NaN                          NaN 49% of errors made at confidence above 0.70. High ethical risk — the model is confidently wrong here.
slang-heavy               23                  0.3833 0.4783 0.7064          Yes         -0.0246    0.0400   0.0255                           No 48% of errors made at confidence above 0.70. High ethical risk — the model is